# IDH1-PRKDC Pair Injection + Subgraph Extraction

This notebook:
1. Loads the main full graph (`/data/guoyu/KG-LLM-XSL/output/ablation_graphs/full.graphml`).
2. Explicitly injects gene pair `IDH1-PRKDC` and augments it using the same graph-module APIs and databases.
3. Extracts the explanation subgraph using algorithm-module APIs.
4. Saves subgraph outputs including a PNG image.

In [21]:
from __future__ import annotations

import copy
import json
import sys
from pathlib import Path
from types import SimpleNamespace

import matplotlib.pyplot as plt
import networkx as nx

cwd = Path.cwd().resolve()
ROOT = cwd if (cwd / "src").exists() else cwd.parent
SRC = ROOT / "src"
for p in (str(SRC), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)

from graph_module import graph_config as gcfg
from graph_module.construction_functions import (
    add_cancer_driver_context,
    add_cbioportal_context,
    add_depmap_context,
    add_open_targets_drugs,
    add_reactome_pathways,
    add_tf_regulation,
    build_base_graph,
    build_logger,
    expand_with_omnipath,
    expand_with_string,
    add_gene,
    node_key,
)
from algorithm_module.graph_search_algo import build_explanation_subgraph, gene_node
from algorithm_module.utils.graph_search_utils import load_graphml
from graph_module.utils.graph_vis import graph_vis

PAIR_A = "IDH1"
PAIR_B = "PRKDC"
FULL_GRAPH_PATH = Path("/data/guoyu/KG-LLM-XSL/output/ablation_graphs/full.graphml")
OUT_DIR = ROOT / "playground" / "output" / f"{PAIR_A}_{PAIR_B}_notebook"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT:", ROOT)
print("FULL_GRAPH_PATH exists:", FULL_GRAPH_PATH.exists())
print("OUT_DIR:", OUT_DIR)

ROOT: /home/guoyu/grad_design/KG-LLM-XSL
FULL_GRAPH_PATH exists: True
OUT_DIR: /home/guoyu/grad_design/KG-LLM-XSL/playground/output/IDH1_PRKDC_notebook


In [22]:
logger = build_logger("idh1-prkdc-notebook")

def make_pair_cfg(pair_a: str, pair_b: str):
    # Clone graph config attributes into an isolated namespace for pair-focused expansion.
    attrs = {}
    for name in dir(gcfg):
        if name.startswith("__"):
            continue
        attrs[name] = getattr(gcfg, name)

    cfg = SimpleNamespace(**attrs)
    pair_a = str(pair_a).strip().upper()
    pair_b = str(pair_b).strip().upper()

    cfg.CORE_GENE_PAIRS = [(pair_a, pair_b)]
    cfg.GENE_PAIRS = [(pair_a, pair_b)]
    # cfg.SL_PAIRS_COMMON = [
    #     {"gene_a": pair_a, "gene_b": pair_b, "context": "notebook_manual_pair", "note": "explicit pair injection"}
    # ]

    # Keep lookup if available; otherwise expansion functions naturally skip unavailable info.
    lookup = dict(getattr(gcfg, "ENSEMBL_LOOKUP", {}) or {})
    cfg.ENSEMBL_LOOKUP = lookup
    return cfg


def build_pair_delta_graph(pair_a: str, pair_b: str) -> nx.MultiDiGraph:
    cfg = make_pair_cfg(pair_a, pair_b)
    g = build_base_graph(cfg=cfg, logger=logger)

    # Explicitly ensure both genes and pair edges are present.
    # ga = add_gene(g, pair_a, source="seed", ensembl_id=(cfg.ENSEMBL_LOOKUP or {}).get(pair_a))
    # gb = add_gene(g, pair_b, source="seed", ensembl_id=(cfg.ENSEMBL_LOOKUP or {}).get(pair_b))
    # g.add_edge(ga, gb, type="SL_pair", source="seed", context="notebook_manual_pair")
    # g.add_edge(gb, ga, type="SL_pair", source="seed", context="notebook_manual_pair")

    # Same expansion order as graph_module.graph_construct.PIPELINE_STEPS
    steps = [
        ("string", expand_with_string),
        ("omnipath", expand_with_omnipath),
        ("tf_regulation", add_tf_regulation),
        ("depmap", add_depmap_context),
        ("reactome", add_reactome_pathways),
        ("cbioportal", add_cbioportal_context),
        ("cancer_driver", add_cancer_driver_context),
        ("opentargets", add_open_targets_drugs),
    ]

    for name, fn in steps:
        try:
            if name == "omnipath":
                fn(g, cfg=cfg, logger=logger, uniprot_client=None)
            else:
                fn(g, cfg=cfg, logger=logger)
        except Exception as e:
            # Keep behavior robust: if a data source is unavailable, continue.
            print(f"[WARN] step={name} skipped due to: {e}")

    return g


def merge_delta_into_full(full_g: nx.MultiDiGraph, delta_g: nx.MultiDiGraph, pair_a: str, pair_b: str) -> dict:
    pair_nodes = {gene_node(pair_a), gene_node(pair_b)}
    # Keep only the component around the injected pair to avoid unrelated additions.
    und = delta_g.to_undirected(as_view=True)
    keep_nodes = set(pair_nodes)
    for pn in pair_nodes:
        if pn in delta_g:
            keep_nodes.add(pn)
            keep_nodes.update(nx.single_source_shortest_path_length(und, pn, cutoff=2).keys())

    small = delta_g.subgraph(keep_nodes).copy()

    added_nodes = 0
    for n, attrs in small.nodes(data=True):
        if not full_g.has_node(n):
            full_g.add_node(n, **(attrs or {}))
            added_nodes += 1
        else:
            for k, v in (attrs or {}).items():
                if k not in full_g.nodes[n] or full_g.nodes[n].get(k) in (None, ""):
                    full_g.nodes[n][k] = v

    added_edges = 0
    for u, v, attrs in small.edges(data=True):
        attrs = attrs or {}
        e_type = str(attrs.get("type") or "")
        e_source = str(attrs.get("source") or "")

        exists = False
        if full_g.has_edge(u, v):
            for _k, eattrs in full_g.get_edge_data(u, v).items():
                if str((eattrs or {}).get("type") or "") == e_type and str((eattrs or {}).get("source") or "") == e_source:
                    exists = True
                    break
        if not exists:
            full_g.add_edge(u, v, **attrs)
            added_edges += 1

    # Ensure explicit pair edge remains present in both directions.
    # ga = gene_node(pair_a)
    # gb = gene_node(pair_b)
    # full_g.add_edge(ga, gb, type="SL_pair", source="seed", context="notebook_manual_pair")
    # full_g.add_edge(gb, ga, type="SL_pair", source="seed", context="notebook_manual_pair")

    return {
        "delta_nodes": int(delta_g.number_of_nodes()),
        "delta_edges": int(delta_g.number_of_edges()),
        "merged_component_nodes": int(small.number_of_nodes()),
        "merged_component_edges": int(small.number_of_edges()),
        "added_nodes": int(added_nodes),
        "added_edges": int(added_edges),
    }

In [23]:
full_graph = load_graphml(FULL_GRAPH_PATH)
print("Loaded full graph:", full_graph.number_of_nodes(), "nodes,", full_graph.number_of_edges(), "edges")

delta_graph = build_pair_delta_graph(PAIR_A, PAIR_B)
merge_stats = merge_delta_into_full(full_graph, delta_graph, PAIR_A, PAIR_B)
print(json.dumps(merge_stats, indent=2))

def _sanitize_for_graphml(g: nx.MultiDiGraph) -> nx.MultiDiGraph:
    g2 = g.copy()
    for _n, _attrs in g2.nodes(data=True):
        for k, v in list((_attrs or {}).items()):
            if v is None or isinstance(v, (str, int, float, bool)):
                continue
            if isinstance(v, (dict, list, tuple, set)):
                _attrs[k] = json.dumps(v, ensure_ascii=False)
            else:
                _attrs[k] = str(v)
    for _u, _v, _attrs in g2.edges(data=True):
        for k, v in list((_attrs or {}).items()):
            if v is None or isinstance(v, (str, int, float, bool)):
                continue
            if isinstance(v, (dict, list, tuple, set)):
                _attrs[k] = json.dumps(v, ensure_ascii=False)
            else:
                _attrs[k] = str(v)
    return g2

aug_graph_path = OUT_DIR / "full_with_IDH1_PRKDC.graphml"
nx.write_graphml(_sanitize_for_graphml(full_graph), aug_graph_path)
print("Augmented graph saved:", aug_graph_path)

Seed graph: 239 nodes, 602 edges


Loaded full graph: 1169 nodes, 3660 edges


STRING: 100%|██████████| 1397/1397 [00:00<00:00, 9817.69it/s]
STRING ready: 249 nodes 1999 edges
UniProt: 20/20 genes cached
OmniPath: 100%|██████████| 150/150 [00:00<00:00, 19127.04it/s]
OmniPath ready: 620 nodes 2409 edges
TF regulation: 100%|██████████| 53/53 [00:00<00:00, 28350.73it/s]
TF regulation ready: 232 edges
DepMap: 100%|██████████| 247/247 [00:00<00:00, 1072.87it/s]
DepMap ready: 11288 edges
Reactome: 247/247 genes cached
Reactome: 2/2 genes cached
Reactome ready
cBioPortal: skipped (stub mode disabled)
IntOGen driver context ready: 102 edges
Dropped cohort TCGA-OV (isolated or connected to all genes)
Dropped cohort TCGA-KIRC (isolated or connected to all genes)
Dropped cohort TCGA-LGG (isolated or connected to all genes)
OpenTargets unreachable (HEAD probe): using cache only
OpenTargets add: 100%|██████████| 75/75 [00:00<00:00, 45876.16it/s]
OpenTargets cache ready


{
  "delta_nodes": 1192,
  "delta_edges": 3631,
  "merged_component_nodes": 386,
  "merged_component_edges": 2443,
  "added_nodes": 6,
  "added_edges": 24
}
Augmented graph saved: /home/guoyu/grad_design/KG-LLM-XSL/playground/output/IDH1_PRKDC_notebook/full_with_IDH1_PRKDC.graphml


In [24]:
sub, meta = build_explanation_subgraph(
    full_graph,
    PAIR_A,
    PAIR_B,
    neigh_max_hops=2,
    max_nodes=None,
)
print("Subgraph nodes/edges:", sub.number_of_nodes(), sub.number_of_edges())
print(json.dumps(meta, indent=2))

Subgraph nodes/edges: 29 34
{
  "gene_a": "gene:IDH1",
  "gene_b": "gene:PRKDC",
  "selected_nodes": 29,
  "selected_edges": 34,
  "paths": [
    {
      "nodes": [
        "gene:IDH1",
        "gene:ATRX",
        "gene:XRCC5",
        "gene:PRKDC"
      ],
      "steps": [
        {
          "src": "gene:IDH1",
          "dst": "gene:ATRX",
          "edge_type": "STRING_association",
          "edge_source": "STRING",
          "direction": "reverse",
          "edge_key": "0",
          "attrs": {
            "type": "STRING_association",
            "source": "STRING"
          }
        },
        {
          "src": "gene:ATRX",
          "dst": "gene:XRCC5",
          "edge_type": "STRING_association",
          "edge_source": "STRING",
          "direction": "forward",
          "edge_key": "0",
          "attrs": {
            "type": "STRING_association",
            "source": "STRING"
          }
        },
        {
          "src": "gene:XRCC5",
          "dst": "gene:PRK

In [25]:
from IPython.display import IFrame, display

sub_graphml = OUT_DIR / "IDH1_PRKDC_subgraph.graphml"
sub_html = OUT_DIR / "IDH1_PRKDC_subgraph.html"

def _sanitize_for_graphml_local(g: nx.MultiDiGraph) -> nx.MultiDiGraph:
    g2 = g.copy()
    for _n, _attrs in g2.nodes(data=True):
        for k, v in list((_attrs or {}).items()):
            if v is None or isinstance(v, (str, int, float, bool)):
                continue
            if isinstance(v, (dict, list, tuple, set)):
                _attrs[k] = json.dumps(v, ensure_ascii=False)
            else:
                _attrs[k] = str(v)
    for _u, _v, _attrs in g2.edges(data=True):
        for k, v in list((_attrs or {}).items()):
            if v is None or isinstance(v, (str, int, float, bool)):
                continue
            if isinstance(v, (dict, list, tuple, set)):
                _attrs[k] = json.dumps(v, ensure_ascii=False)
            else:
                _attrs[k] = str(v)
    return g2

nx.write_graphml(_sanitize_for_graphml_local(sub), sub_graphml)
graph_vis(sub, sub_html, title="IDH1-PRKDC subgraph")

print("Saved:")
print(" -", sub_graphml)
print(" -", sub_html)


Saved:
 - /home/guoyu/grad_design/KG-LLM-XSL/playground/output/IDH1_PRKDC_notebook/IDH1_PRKDC_subgraph.graphml
 - /home/guoyu/grad_design/KG-LLM-XSL/playground/output/IDH1_PRKDC_notebook/IDH1_PRKDC_subgraph.html


In [26]:
# Build the 1-hop incident neighborhood around IDH1 or PRKDC from the augmented full graph.
# NOTE: This cell only uses local files and in-repo APIs; it does not fetch remote data.

from IPython.display import IFrame, display

AUG_GRAPH_PATH = OUT_DIR / "full_with_IDH1_PRKDC.graphml"
if not AUG_GRAPH_PATH.exists():
    raise FileNotFoundError(f"Expected augmented graph at {AUG_GRAPH_PATH}, but it does not exist. Run the earlier save step first.")

g_aug = load_graphml(AUG_GRAPH_PATH)
center_nodes = {gene_node(PAIR_A), gene_node(PAIR_B)}

def sanitize_for_graphml(g: nx.MultiDiGraph) -> nx.MultiDiGraph:
    g2 = g.copy()
    for _n, _attrs in g2.nodes(data=True):
        for k, v in list((_attrs or {}).items()):
            if v is None or isinstance(v, (str, int, float, bool)):
                continue
            if isinstance(v, (dict, list, tuple, set)):
                _attrs[k] = json.dumps(v, ensure_ascii=False)
            else:
                _attrs[k] = str(v)
    for _u, _v, _attrs in g2.edges(data=True):
        for k, v in list((_attrs or {}).items()):
            if v is None or isinstance(v, (str, int, float, bool)):
                continue
            if isinstance(v, (dict, list, tuple, set)):
                _attrs[k] = json.dumps(v, ensure_ascii=False)
            else:
                _attrs[k] = str(v)
    return g2

def incident_neighborhood(g: nx.MultiDiGraph, centers: set[str]) -> nx.MultiDiGraph:
    out = nx.MultiDiGraph()
    # Add all incident edges (u->v) where u or v is a center node.
    for u, v, k, attrs in g.edges(keys=True, data=True):
        if u not in centers and v not in centers:
            continue
        if not out.has_node(u):
            out.add_node(u, **(g.nodes[u] if u in g else {}))
        if not out.has_node(v):
            out.add_node(v, **(g.nodes[v] if v in g else {}))
        out.add_edge(u, v, key=k, **(attrs or {}))
    return out

nbr = incident_neighborhood(g_aug, center_nodes)
print("Augmented graph:", g_aug.number_of_nodes(), "nodes,", g_aug.number_of_edges(), "edges")
print("Incident neighborhood:", nbr.number_of_nodes(), "nodes,", nbr.number_of_edges(), "edges")

nbr_graphml = OUT_DIR / "IDH1_PRKDC_incident_1hop.graphml"
nbr_html = OUT_DIR / "IDH1_PRKDC_incident_1hop.html"
nx.write_graphml(sanitize_for_graphml(nbr), nbr_graphml)
graph_vis(nbr, nbr_html, title="IDH1-PRKDC 1-hop incident neighborhood")

print("Saved:")
print(" -", nbr_graphml)
print(" -", nbr_html)


Augmented graph: 1175 nodes, 3684 edges
Incident neighborhood: 55 nodes, 66 edges
Saved:
 - /home/guoyu/grad_design/KG-LLM-XSL/playground/output/IDH1_PRKDC_notebook/IDH1_PRKDC_incident_1hop.graphml
 - /home/guoyu/grad_design/KG-LLM-XSL/playground/output/IDH1_PRKDC_notebook/IDH1_PRKDC_incident_1hop.html
